In [1]:
import os
import anndata as ad
import numpy as np
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import omicverse as ov
import scvi
from scvi.model.utils import mde

import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2


   ____            _     _    __                  
  / __ \____ ___  (_)___| |  / /__  _____________ 
 / / / / __ `__ \/ / ___/ | / / _ \/ ___/ ___/ _ \ 
/ /_/ / / / / / / / /__ | |/ /  __/ /  (__  )  __/ 
\____/_/ /_/ /_/_/\___/ |___/\___/_/  /____/\___/                                              

Version: 1.5.3, Tutorials: https://omicverse.readthedocs.io/


[rank: 0] Global seed set to 0


In [2]:
sc.settings.set_figure_params(dpi=100, frameon=False)
sc.set_figure_params(dpi=100)
sc.set_figure_params(figsize=(3, 3))
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (3, 3)

In [3]:
# Change the working directory to the Garfield folder (if needed)
os.chdir('/storage2/liuxiaodongLab/fanxueying/embryo_benchmarking_rebuttal/code/20250812_label_transfer_method_quantification_v3')
os.getcwd()

'/storage2/liuxiaodongLab/fanxueying/embryo_benchmarking_rebuttal/code/20250812_label_transfer_method_quantification_v3'

In [4]:
# Process query datasets from folder
query_folder = '/storage2/liuxiaodongLab/fanxueying/mayanalysis/2024Aug/garfield/in_vitro_embryo_models/processed/data'
# List all h5ad files in the directory
h5ad_files = [os.path.join(query_folder, f) for f in os.listdir(query_folder) if f.startswith('corrected_processed_')]

In [5]:
h5ad_files

['/storage2/liuxiaodongLab/fanxueying/mayanalysis/2024Aug/garfield/in_vitro_embryo_models/processed/data/corrected_processed_Weatherbee.h5ad',
 '/storage2/liuxiaodongLab/fanxueying/mayanalysis/2024Aug/garfield/in_vitro_embryo_models/processed/data/corrected_processed_Liu.h5ad',
 '/storage2/liuxiaodongLab/fanxueying/mayanalysis/2024Aug/garfield/in_vitro_embryo_models/processed/data/corrected_processed_Rowan.h5ad',
 '/storage2/liuxiaodongLab/fanxueying/mayanalysis/2024Aug/garfield/in_vitro_embryo_models/processed/data/corrected_processed_zheng_2019.h5ad',
 '/storage2/liuxiaodongLab/fanxueying/mayanalysis/2024Aug/garfield/in_vitro_embryo_models/processed/data/corrected_processed_Ai_model.h5ad',
 '/storage2/liuxiaodongLab/fanxueying/mayanalysis/2024Aug/garfield/in_vitro_embryo_models/processed/data/corrected_processed_Oldak.h5ad',
 '/storage2/liuxiaodongLab/fanxueying/mayanalysis/2024Aug/garfield/in_vitro_embryo_models/processed/data/corrected_processed_Hislop.h5ad',
 '/storage2/liuxiaodon

In [6]:
# Read each h5ad file into an AnnData object
adata_list = []
for file in h5ad_files:
    print(f"Reading file: {file}")
    adata = sc.read_h5ad(file)  # Read the h5ad file into AnnData
    adata_list.append(adata)  # Add AnnData object to the list

Reading file: /storage2/liuxiaodongLab/fanxueying/mayanalysis/2024Aug/garfield/in_vitro_embryo_models/processed/data/corrected_processed_Weatherbee.h5ad
Reading file: /storage2/liuxiaodongLab/fanxueying/mayanalysis/2024Aug/garfield/in_vitro_embryo_models/processed/data/corrected_processed_Liu.h5ad
Reading file: /storage2/liuxiaodongLab/fanxueying/mayanalysis/2024Aug/garfield/in_vitro_embryo_models/processed/data/corrected_processed_Rowan.h5ad
Reading file: /storage2/liuxiaodongLab/fanxueying/mayanalysis/2024Aug/garfield/in_vitro_embryo_models/processed/data/corrected_processed_zheng_2019.h5ad
Reading file: /storage2/liuxiaodongLab/fanxueying/mayanalysis/2024Aug/garfield/in_vitro_embryo_models/processed/data/corrected_processed_Ai_model.h5ad
Reading file: /storage2/liuxiaodongLab/fanxueying/mayanalysis/2024Aug/garfield/in_vitro_embryo_models/processed/data/corrected_processed_Oldak.h5ad
Reading file: /storage2/liuxiaodongLab/fanxueying/mayanalysis/2024Aug/garfield/in_vitro_embryo_models

In [7]:
# add scpoli lineage results
# Define the output directory
output_dir = './embryo_model_scPoli/lineage'

# Step 1: List all CSV files in the directory
scpoli_files = [os.path.join(output_dir, f) for f in os.listdir(output_dir) if f.endswith('.csv')]

# Step 2: Initialize an empty dictionary to store the results
scpoli_tables = {}

# Step 3: Process each file
for file in scpoli_files:
    # Read the CSV file into a DataFrame
    scpoli = pd.read_csv(file)
    
    # Extract the desired substring from the file name
    file_name = os.path.basename(file)  # Get the file name without the path
    new_name = file_name.replace('corrected_processed_', '')  # Remove "corrected_processed_"
    new_name = new_name.replace('.h5ad_scPoli_query.csv', '')  # Remove ".h5ad"
    
    # Store the DataFrame in the dictionary with the new name
    scpoli_tables[new_name] = scpoli

# Now `scpoli_tables` is a dictionary where keys are the processed file names and values are the DataFrames

# List of attributes to be extracted from Garfield results
attri = ["lineage_pred", "lineage_uncert"]

# Process each AnnData object and merge with Garfield results
for i, dataset in enumerate(adata_list):
    # Extract the dataset name from the file name
    name = os.path.basename(h5ad_files[i]).replace('.h5ad', '')  
    name = name.replace('corrected_processed_', '')  
    
    # Get the corresponding scpoli results
    garf = scpoli_tables.get(name)
    
    if garf is not None:
        # Check if 'X' is the first column or index
        if garf.columns[0] != 'X':  
            print(f"Warning: The first column is not 'X' for {name}. Renaming the first column.")
            garf.rename(columns={garf.columns[0]: 'X'}, inplace=True)
        
        # Set the first column as index
        garf.set_index('X', inplace=True)
        garf.index = garf.index.str.replace('-1-1$', '-1', regex=True)  # Fix extra "-1"

        # Check if indices match
        if not garf.index.equals(dataset.obs_names):
            print(f"Warning: Indices do not match for {name}. Attempting to fix...")
            
            # Standardize index formatting
            garf.index = garf.index.astype(str)  # Ensure string format
            dataset.obs_names = dataset.obs_names.astype(str)
            
            # Try stripping potential "-1" suffix
            garf.index = garf.index.str.replace('-1$', '', regex=True)
            dataset.obs_names = dataset.obs_names.str.replace('-1$', '', regex=True)
            
            # Check again after fixing
            if not garf.index.equals(dataset.obs_names):
                print(f"Error: Indices still do not match for {name}. Skipping this dataset.")
                continue

        # Extract required columns
        garf = garf[attri]
        
        # Remove filtered cells
        dataset = dataset[dataset.obs_names.isin(garf.index), :]
        
        # Ensure row names match
        garf = garf.loc[dataset.obs_names, :]
        
        # Add prefix to column names
        garf.columns = [f"human_ref_{col}" for col in garf.columns]

        # Merge into AnnData metadata
        dataset.obs = pd.concat([dataset.obs, garf], axis=1)

        # Update dataset in the list
        adata_list[i] = dataset
    else:
        print(f"No scpoli data found for {name}")


In [8]:
# add scpoli cell type results
# Define the output directory
output_dir = './embryo_model_scPoli/reanno'

# Step 1: List all CSV files in the directory
scpoli_files = [os.path.join(output_dir, f) for f in os.listdir(output_dir) if f.endswith('.csv')]

# Step 2: Initialize an empty dictionary to store the results
scpoli_tables = {}

# Step 3: Process each file
for file in scpoli_files:
    # Read the CSV file into a DataFrame
    scpoli = pd.read_csv(file)
    
    # Extract the desired substring from the file name
    file_name = os.path.basename(file)  # Get the file name without the path
    new_name = file_name.replace('corrected_processed_', '')  # Remove "corrected_processed_"
    new_name = new_name.replace('.h5ad_scPoli_query.csv', '')  # Remove ".h5ad"
    
    # Store the DataFrame in the dictionary with the new name
    scpoli_tables[new_name] = scpoli

# Now `scpoli_tables` is a dictionary where keys are the processed file names and values are the DataFrames

# List of attributes to be extracted from Garfield results
attri = ["reanno_pred", "reanno_uncert"]

# Process each AnnData object and merge with Garfield results
for i, dataset in enumerate(adata_list):
    # Extract the dataset name from the file name
    name = os.path.basename(h5ad_files[i]).replace('.h5ad', '')  
    name = name.replace('corrected_processed_', '')  
    
    # Get the corresponding scpoli results
    garf = scpoli_tables.get(name)
    
    if garf is not None:
        # Check if 'X' is the first column or index
        if garf.columns[0] != 'X':  
            print(f"Warning: The first column is not 'X' for {name}. Renaming the first column.")
            garf.rename(columns={garf.columns[0]: 'X'}, inplace=True)
        
        # Set the first column as index
        garf.set_index('X', inplace=True)
        garf.index = garf.index.str.replace('-1-1$', '-1', regex=True)  # Fix extra "-1"

        # Check if indices match
        if not garf.index.equals(dataset.obs_names):
            print(f"Warning: Indices do not match for {name}. Attempting to fix...")
            
            # Standardize index formatting
            garf.index = garf.index.astype(str)  # Ensure string format
            dataset.obs_names = dataset.obs_names.astype(str)
            
            # Try stripping potential "-1" suffix
            garf.index = garf.index.str.replace('-1$', '', regex=True)
            dataset.obs_names = dataset.obs_names.str.replace('-1$', '', regex=True)
            
            # Check again after fixing
            if not garf.index.equals(dataset.obs_names):
                print(f"Error: Indices still do not match for {name}. Skipping this dataset.")
                continue

        # Extract required columns
        garf = garf[attri]
        
        # Remove filtered cells
        dataset = dataset[dataset.obs_names.isin(garf.index), :]
        
        # Ensure row names match
        garf = garf.loc[dataset.obs_names, :]
        
        # Add prefix to column names
        garf.columns = [f"human_ref_{col}" for col in garf.columns]

        # Merge into AnnData metadata
        dataset.obs = pd.concat([dataset.obs, garf], axis=1)

        # Update dataset in the list
        adata_list[i] = dataset
    else:
        print(f"No scpoli data found for {name}")


In [9]:
# add mapquery results
# Define the output directory
output_dir = './embryo_model_querymap'

# Step 1: List all CSV files in the directory
querymap_files = [os.path.join(output_dir, f) for f in os.listdir(output_dir) if f.endswith('.csv')]

# Step 2: Initialize an empty dictionary to store the results
querymap_tables = {}

# Step 3: Process each file
for file in querymap_files:
    # Read the CSV file into a DataFrame
    querymap = pd.read_csv(file)
    
    # Extract the desired substring from the file name
    file_name = os.path.basename(file)  # Get the file name without the path
    new_name = file_name.replace('_human_ref_querymap.csv', '')  # Remove "corrected_processed_"
    
    # Store the DataFrame in the dictionary with the new name
    querymap_tables[new_name] = querymap

# Now `scgpt_tables` is a dictionary where keys are the processed file names and values are the DataFrames

# List of attributes to be extracted from Garfield results
attri = ["querymap_human_ref_lineage", "querymap_human_ref_reanno"]

# Process each AnnData object and merge with Garfield results
for i, dataset in enumerate(adata_list):
    # Extract the dataset name from the file name (assuming `garfield_tables` keys are the file names without paths)
    name = os.path.basename(h5ad_files[i]).replace('.h5ad', '')  # Remove the file extension
    name = name.replace('corrected_processed_', '')  # Remove the file prefix
    
    # Get the corresponding Garfield results
    garf = querymap_tables.get(name)
    
    if garf is not None:
        # Check if 'X' is the first column or index
        if garf.columns[0] != 'X':  # If the first column is not 'X'
            print(f"Warning: The first column is not 'X' for {name}. Renaming the first column.")
            garf.rename(columns={garf.columns[0]: 'X'}, inplace=True)
        
        # Set the first column as index (row names)
        garf.set_index('X', inplace=True)
        
        # Extract the required columns from Garfield table
        garf = garf[attri]
        
        # Remove filtered cells from dataset
        dataset = dataset[dataset.obs_names.isin(garf.index), :]
        
        # Ensure row names in dataset and Garfield data match
        garf = garf.loc[dataset.obs_names, :]
        
        # Add prefix to Garfield column names
        garf.columns = [f"human_ref_{col}" for col in garf.columns]

        # Merge Garfield results into the AnnData metadata
        dataset.obs = pd.concat([dataset.obs, garf], axis=1)

        # Update the dataset in the list
        adata_list[i] = dataset
    else:
        print(f"No Garfield data found for {name}")


In [10]:
# add scgpt results
# Define the output directory
output_dir = './embryo_model_scGPT'

# Step 1: List all CSV files in the directory
scgpt_files = [os.path.join(output_dir, f) for f in os.listdir(output_dir) if f.endswith('.csv')]

# Step 2: Initialize an empty dictionary to store the results
scgpt_tables = {}

# Step 3: Process each file
for file in scgpt_files:
    # Read the CSV file into a DataFrame
    scgpt = pd.read_csv(file)
    
    # Extract the desired substring from the file name
    file_name = os.path.basename(file)  # Get the file name without the path
    new_name = file_name.replace('human_', '')  # Remove "corrected_processed_"
    new_name = new_name.replace('_scgpt.csv', '')  # Remove ".h5ad"
    
    # Store the DataFrame in the dictionary with the new name
    scgpt_tables[new_name] = scgpt

# Now `scgpt_tables` is a dictionary where keys are the processed file names and values are the DataFrames
# List of attributes to be extracted from Garfield results
attri = ["scgpt_final_lineage_pre", "scgpt_final_anno_pre"]

# Process each AnnData object and merge with Garfield results
for i, dataset in enumerate(adata_list):
    # Extract the dataset name from the file name (assuming `garfield_tables` keys are the file names without paths)
    name = os.path.basename(h5ad_files[i]).replace('.h5ad', '')  # Remove the file extension
    name = name.replace('corrected_processed_', '')  # Remove the file prefix
    
    # Get the corresponding Garfield results
    garf = scgpt_tables.get(name)
    
    if garf is not None:
        # Check if 'X' is the first column or index
        if garf.columns[0] != 'X':  # If the first column is not 'X'
            print(f"Warning: The first column is not 'X' for {name}. Renaming the first column.")
            garf.rename(columns={garf.columns[0]: 'X'}, inplace=True)
        
        # Set the first column as index (row names)
        garf.set_index('X', inplace=True)
        
        # Extract the required columns from Garfield table
        garf = garf[attri]
        
        # Remove filtered cells from dataset
        dataset = dataset[dataset.obs_names.isin(garf.index), :]
        
        # Ensure row names in dataset and Garfield data match
        garf = garf.loc[dataset.obs_names, :]
        
        # Add prefix to Garfield column names
        garf.columns = [f"human_ref_{col}" for col in garf.columns]

        # Merge Garfield results into the AnnData metadata
        dataset.obs = pd.concat([dataset.obs, garf], axis=1)

        # Update the dataset in the list
        adata_list[i] = dataset
    else:
        print(f"No Garfield data found for {name}")


In [11]:
# add scarches results
# Define the output directory
output_dir = './embryo_model_scArches'

# Step 1: List all CSV files in the directory
scarches_files = [os.path.join(output_dir, f) for f in os.listdir(output_dir) if f.endswith('.csv')]

# Step 2: Initialize an empty dictionary to store the results
scarches_tables = {}

# Step 3: Process each file
for file in scarches_files:
    # Read the CSV file into a DataFrame
    scarches = pd.read_csv(file)
    
    # Extract the desired substring from the file name
    file_name = os.path.basename(file)  # Get the file name without the path
    new_name = file_name.replace('human_', '')  # Remove "corrected_processed_"
    new_name = new_name.replace('_scArches.csv', '')  # Remove ".h5ad"
    
    # Store the DataFrame in the dictionary with the new name
    scarches_tables[new_name] = scarches

# Now `scpoli_tables` is a dictionary where keys are the processed file names and values are the DataFrames

# List of attributes to be extracted from Garfield results
attri = ["scArches_final_anno_pre", "scArches_final_anno_uncertainty","scArches_final_lineage_pre", "scArches_final_lineage_uncertainty"]

# Process each AnnData object and merge with Garfield results
for i, dataset in enumerate(adata_list):
    # Extract the dataset name from the file name (assuming `garfield_tables` keys are the file names without paths)
    name = os.path.basename(h5ad_files[i]).replace('.h5ad', '')  # Remove the file extension
    name = name.replace('corrected_processed_', '')  # Remove the file prefix
    
    # Get the corresponding Garfield results
    garf = scarches_tables.get(name)
    
    if garf is not None:
        # Check if 'X' is the first column or index
        if garf.columns[0] != 'X':  # If the first column is not 'X'
            print(f"Warning: The first column is not 'X' for {name}. Renaming the first column.")
            garf.rename(columns={garf.columns[0]: 'X'}, inplace=True)
        
        # Set the first column as index (row names)
        garf.set_index('X', inplace=True)
        
        # Extract the required columns from Garfield table
        garf = garf[attri]
        
        # Remove filtered cells from dataset
        dataset = dataset[dataset.obs_names.isin(garf.index), :]
        
        # Ensure row names in dataset and Garfield data match
        garf = garf.loc[dataset.obs_names, :]
        
        # Add prefix to Garfield column names
        garf.columns = [f"human_ref_{col}" for col in garf.columns]

        # Merge Garfield results into the AnnData metadata
        dataset.obs = pd.concat([dataset.obs, garf], axis=1)

        # Update the dataset in the list
        adata_list[i] = dataset
    else:
        print(f"No Garfield data found for {name}")



In [12]:
# Define the output directory
output_dir = './embryo_model_garfield'

# Step 1: List all CSV files in the directory
garfield_files = [os.path.join(output_dir, f) for f in os.listdir(output_dir) if f.endswith('.csv')]

# Step 2: Initialize an empty dictionary to store the results
garfield_tables = {}

# Step 3: Process each file
for file in garfield_files:
    # Read the CSV file into a DataFrame
    garfield = pd.read_csv(file)
    
    # Extract the desired substring from the file name
    file_name = os.path.basename(file)  # Get the file name without the path
    new_name = file_name.replace('corrected_processed_', '')  # Remove "corrected_processed_"
    new_name = new_name.replace('.h5ad', '')  # Remove ".h5ad"
    new_name = new_name.split('_garfield')[0]  # Remove "_garfield" and anything after
    
    # Store the DataFrame in the dictionary with the new name
    garfield_tables[new_name] = garfield

# Now `garfield_tables` is a dictionary where keys are the processed file names and values are the DataFrames

# List of attributes to be extracted from Garfield results
attri = ["transferred_reanno_unfiltered", "transferred_reanno_uncert", 
         "transferred_lineage_unfiltered", "transferred_lineage_uncert"]

# Process each AnnData object and merge with Garfield results
for i, dataset in enumerate(adata_list):
    # Extract the dataset name from the file name (assuming `garfield_tables` keys are the file names without paths)
    name = os.path.basename(h5ad_files[i]).replace('.h5ad', '')  # Remove the file extension
    name = name.replace('corrected_processed_', '')  # Remove the file prefix
    
    # Get the corresponding Garfield results
    garf = garfield_tables.get(name)
    
    if garf is not None:
        # Check if 'X' is the first column or index
        if garf.columns[0] != 'X':  # If the first column is not 'X'
            print(f"Warning: The first column is not 'X' for {name}. Renaming the first column.")
            garf.rename(columns={garf.columns[0]: 'X'}, inplace=True)
        
        # Set the first column as index (row names)
        garf.set_index('X', inplace=True)
        
        # Extract the required columns from Garfield table
        garf = garf[attri]
        
        # Remove filtered cells from dataset
        dataset = dataset[dataset.obs_names.isin(garf.index), :]
        
        # Ensure row names in dataset and Garfield data match
        garf = garf.loc[dataset.obs_names, :]
        
        # Add prefix to Garfield column names
        garf.columns = [f"human_ref_{col}" for col in garf.columns]

        # Merge Garfield results into the AnnData metadata
        dataset.obs = pd.concat([dataset.obs, garf], axis=1)

        # Update the dataset in the list
        adata_list[i] = dataset
    else:
        print(f"No Garfield data found for {name}")


In [13]:
import os

# Loop through each dataset in adata_list
for i, dataset in enumerate(adata_list):
    # Get the dataset name from the h5ad_files list (assuming it's parallel)
    name = os.path.basename(h5ad_files[i]).replace('.h5ad', '').replace('corrected_processed_', '')
    
    print(f"\n{'='*50}")
    print(f"Observation metadata (obs) for dataset: {name}")
    print(f"{'='*50}")
    
    # Print the first few rows of obs
    print(dataset.obs.head())  # You can use .tail() or entire dataset if needed


Observation metadata (obs) for dataset: Weatherbee
                      orig.ident  nCount_RNA  nFeature_RNA  percent.mt  \
AAACAGCCAATATACC-1_1  Weatherbee      7994.0          3257    4.953715   
AAACAGCCACTAAGCC-1_1  Weatherbee      8411.0          3593    4.541672   
AAACATGCAGTATGTT-1_1  Weatherbee      7937.0          3497    3.187602   
AAACCAACAATAACCT-1_1  Weatherbee     10750.0          4132    2.483721   
AAACCGAAGGCTAGAA-1_1  Weatherbee     10730.0          4098    3.122088   

                                sample_type scmap_nakamura  \
AAACAGCCAATATACC-1_1  double_structure_day8      Post-paTE   
AAACAGCCACTAAGCC-1_1  double_structure_day8      Post-paTE   
AAACATGCAGTATGTT-1_1  double_structure_day8           EXMC   
AAACCAACAATAACCT-1_1  double_structure_day8           EXMC   
AAACCGAAGGCTAGAA-1_1  double_structure_day8           EXMC   

                                 scmapCELL_Yang scmap_ma  scmap_Tyser  \
AAACAGCCAATATACC-1_1                 unassigned    L-AM2 

In [14]:
# Define the mapping of old column names to new column names
columns_to_rename = {
    "human_ref_lineage_pred": "scpoli_lineage_pred",
    "human_ref_lineage_uncert": "scpoli_lineage_uncert",
    "human_ref_reanno_pred": "scpoli_reanno_pred",
    "human_ref_reanno_uncert": "scpoli_reanno_uncert",
    "human_ref_querymap_human_ref_lineage": "querymap_lineage_pred",
    "human_ref_querymap_human_ref_reanno": "querymap_reanno_pred",
    "human_ref_scgpt_final_lineage_pre": "scgpt_lineage_pred",
    "human_ref_scgpt_final_anno_pre": "scgpt_reanno_pred",
    "human_ref_scArches_final_anno_pre": "scArches_reanno_pred",
    "human_ref_scArches_final_anno_uncertainty": "scArches_reanno_uncert",
    "human_ref_scArches_final_lineage_pre": "scArches_lineage_pred",
    "human_ref_scArches_final_lineage_uncertainty": "scArches_lineage_uncert",
    "human_ref_transferred_reanno_unfiltered": "garfield_reanno_pred",
    "human_ref_transferred_reanno_uncert": "garfield_reanno_uncert",
    "human_ref_transferred_lineage_unfiltered": "garfield_lineage_pred",
    "human_ref_transferred_lineage_uncert": "garfield_lineage_uncert"
}

# Loop through each AnnData object in adata_list
for i, dataset in enumerate(adata_list):
    # Rename the columns in .obs
    dataset.obs.rename(columns=columns_to_rename, inplace=True)
    
    # Optional: Update the adata_list with the modified dataset
    adata_list[i] = dataset

    # Optional: Print confirmation for each dataset
    print(f"Renamed columns in dataset {i+1} (e.g., 'lineage_pred' → 'scpoli_lineage_pred')")

for i, dataset in enumerate(adata_list):
    name = os.path.basename(h5ad_files[i]).replace('.h5ad', '').replace('corrected_processed_', '')
    print(f"\nObservation metadata (obs) for dataset: {name}")
    print(dataset.obs.columns.tolist())

Renamed columns in dataset 1 (e.g., 'lineage_pred' → 'scpoli_lineage_pred')
Renamed columns in dataset 2 (e.g., 'lineage_pred' → 'scpoli_lineage_pred')
Renamed columns in dataset 3 (e.g., 'lineage_pred' → 'scpoli_lineage_pred')
Renamed columns in dataset 4 (e.g., 'lineage_pred' → 'scpoli_lineage_pred')
Renamed columns in dataset 5 (e.g., 'lineage_pred' → 'scpoli_lineage_pred')
Renamed columns in dataset 6 (e.g., 'lineage_pred' → 'scpoli_lineage_pred')
Renamed columns in dataset 7 (e.g., 'lineage_pred' → 'scpoli_lineage_pred')
Renamed columns in dataset 8 (e.g., 'lineage_pred' → 'scpoli_lineage_pred')
Renamed columns in dataset 9 (e.g., 'lineage_pred' → 'scpoli_lineage_pred')

Observation metadata (obs) for dataset: Weatherbee
['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'sample_type', 'scmap_nakamura', 'scmapCELL_Yang', 'scmap_ma', 'scmap_Tyser', 'scmapCELL_Mole', 'cell_assignment', 'course_cell_assignment', 'stage', 'species', 'embryo', 'platform', 'doublet', 'doublet_sc

In [15]:
def preprocess_data(query_adata: sc.AnnData) -> sc.AnnData:
    """
    Preprocess the dataset and compute PCA and UMAP embeddings.
    """
    print("[INFO] Starting preprocessing...")

    # Normalization and log transformation
    sc.pp.normalize_total(query_adata, target_sum=1e4)
    sc.pp.log1p(query_adata)
    query_adata.layers["logcounts"] = query_adata.X.copy()

    # Highly variable gene selection
    sc.pp.highly_variable_genes(
        query_adata,
        n_top_genes=2000,
        flavor="cell_ranger",
        batch_key="orig.ident"
    )

    # PCA
    sc.tl.pca(query_adata, n_comps=30, use_highly_variable=True)

    # Neighbors graph (required for UMAP and clustering)
    sc.pp.neighbors(query_adata, use_rep="X_pca", n_pcs=30)

    # UMAP
    sc.tl.leiden(query_adata, resolution=0.5, key_added="res_0.5", random_state=42)
    sc.tl.umap(query_adata)

    print("[INFO] Preprocessing completed.")
    return query_adata

In [16]:
# Apply preprocessing to each AnnData object in adata_list
preprocessed_list = []

for i, adata in enumerate(adata_list):
    print(f"\n{'='*50}\nProcessing dataset {i+1}: {adata.obs['orig.ident'].unique()[0]}\n{'='*50}")
    
    # Run preprocessing
    processed_adata = preprocess_data(adata)
    
    # Append to new list
    preprocessed_list.append(processed_adata)

print("\n✅ All datasets have been preprocessed.")


Processing dataset 1: Weatherbee
[INFO] Starting preprocessing...
[INFO] Preprocessing completed.

Processing dataset 2: Liu
[INFO] Starting preprocessing...
[INFO] Preprocessing completed.

Processing dataset 3: Rowan
[INFO] Starting preprocessing...
[INFO] Preprocessing completed.

Processing dataset 4: Zheng_2019
[INFO] Starting preprocessing...
[INFO] Preprocessing completed.

Processing dataset 5: Ai_model
[INFO] Starting preprocessing...
[INFO] Preprocessing completed.

Processing dataset 6: Oldak
[INFO] Starting preprocessing...
[INFO] Preprocessing completed.

Processing dataset 7: Hislop
[INFO] Starting preprocessing...
[INFO] Preprocessing completed.

Processing dataset 8: Pedroza
[INFO] Starting preprocessing...
[INFO] Preprocessing completed.

Processing dataset 9: zheng_2022
[INFO] Starting preprocessing...
[INFO] Preprocessing completed.

✅ All datasets have been preprocessed.


In [17]:
import traceback
import time

def calculate_bio_conservation(adata, umap_config, batch_key="orig.ident", 
                              compute_isolated_labels=True, 
                              subsample=None):
    """
    Calculate biological conservation metrics for different UMAP embeddings
    
    Parameters:
    -----------
    adata: AnnData object
    umap_config: Dictionary mapping UMAP keys to list of (lineage_key, cluster_key) tuples
    batch_key: Key for batch information in adata.obs (default: "orig.ident")
    compute_isolated_labels: Whether to compute isolated label metrics (slower)
    subsample: Number of cells to subsample for faster computation (None = use all)
    
    Returns:
    --------
    DataFrame with metrics for each UMAP version
    """
    
    # Initialize results dictionary
    results = {
        'UMAP': [],
        'Lineage_key': [],
        'Cluster_key': [],
        'Silhouette': [],
        'NMI': [],
        'ARI': []
    }
    
    if compute_isolated_labels:
        results['Isolated_label_F1'] = []
        results['Isolated_label_ASW'] = []
    
    # Check if batch_key exists in adata.obs
    if batch_key not in adata.obs:
        print(f"Warning: Batch key '{batch_key}' not found in adata.obs. Some metrics may fail.")
    
    # ✅ NEW LOOP STARTS HERE
    for umap_key in umap_config:
        for (label_key, cluster_key) in umap_config[umap_key]:
            start_time = time.time()
            
            # Check if UMAP key exists in adata.obsm
            if umap_key not in adata.obsm:
                print(f"Warning: UMAP key '{umap_key}' not found in adata.obsm. Skipping.")
                continue

            # Check if label_key exists in adata.obs
            if label_key not in adata.obs:
                print(f"Warning: Label key '{label_key}' not found in adata.obs. Skipping.")
                continue

            # Check if cluster_key exists in adata.obs
            if cluster_key not in adata.obs:
                print(f"Warning: Cluster key '{cluster_key}' not found in adata.obs. Skipping.")
                continue

            print(f"Processing {umap_key} with lineage: {label_key}, clusters: {cluster_key}")
            results['UMAP'].append(umap_key)
            results['Lineage_key'].append(label_key)
            results['Cluster_key'].append(cluster_key)

            # Filter out cells with missing or NA labels or clusters
            mask = ~(adata.obs[label_key].isna() | adata.obs[cluster_key].isna())
            if batch_key in adata.obs:
                mask = mask & ~(adata.obs[batch_key].isna())
            adata_filtered = adata[mask].copy()

            # Subsample if needed (for speed)
            if subsample is not None and subsample < adata_filtered.n_obs:
                print(f"  Subsampling from {adata_filtered.n_obs} to {subsample} cells")
                sc.pp.subsample(adata_filtered, n_obs=subsample, random_state=42)

            print(f"  Working with {adata_filtered.n_obs} cells")

            # Create a temporary AnnData with the correct UMAP key
            temp_adata = adata_filtered.copy()
            temp_adata.obsm['umap'] = temp_adata.obsm[umap_key].copy()

            # Calculate Silhouette score
            try:
                sil_score = scib.metrics.silhouette(temp_adata, label_key=label_key, embed='umap')
                results['Silhouette'].append(sil_score)
                print(f"  Silhouette score: {sil_score}")
            except Exception as e:
                print(f"  Error calculating silhouette for {umap_key}: {e}")
                results['Silhouette'].append(np.nan)

            # Calculate NMI and ARI
            try:
                nmi_score = scib.metrics.nmi(adata_filtered, cluster_key=cluster_key, label_key=label_key)
                results['NMI'].append(nmi_score)
                print(f"  NMI score: {nmi_score}")
            except Exception as e:
                print(f"  Error calculating NMI for {umap_key}: {e}")
                results['NMI'].append(np.nan)

            try:
                ari_score = scib.metrics.ari(adata_filtered, cluster_key=cluster_key, label_key=label_key)
                results['ARI'].append(ari_score)
                print(f"  ARI score: {ari_score}")
            except Exception as e:
                print(f"  Error calculating ARI for {umap_key}: {e}")
                results['ARI'].append(np.nan)

            # Isolated label metrics (optional)
            if compute_isolated_labels:
                print(f"  Computing isolated label metrics (may take a while)...")
                try:
                    iso_f1 = scib.metrics.isolated_labels_f1(
                        temp_adata, 
                        label_key=label_key,
                        batch_key=batch_key,
                        embed='umap',
                        iso_threshold=20,
                        verbose=False
                    )
                    results['Isolated_label_F1'].append(iso_f1)
                    print(f"  Isolated label F1 score: {iso_f1}")
                except Exception as e:
                    print(f"  Error calculating isolated labels F1 for {umap_key}: {e}")
                    results['Isolated_label_F1'].append(np.nan)

                try:
                    iso_asw = scib.metrics.isolated_labels_asw(
                        temp_adata, 
                        label_key=label_key,
                        batch_key=batch_key,
                        embed='umap',
                        iso_threshold=20,
                        verbose=False
                    )
                    results['Isolated_label_ASW'].append(iso_asw)
                    print(f"  Isolated label ASW score: {iso_asw}")
                except Exception as e:
                    print(f"  Error calculating isolated labels ASW for {umap_key}: {e}")
                    results['Isolated_label_ASW'].append(np.nan)

            elapsed_time = time.time() - start_time
            print(f"  Completed in {elapsed_time:.2f} seconds")
    
    return pd.DataFrame(results)

In [18]:

import scib

# List to store results for each dataset
all_metrics_dfs = []

# Loop through each AnnData in the list
for i, adata in enumerate(adata_list):
    print(f"\n{'='*60}\nProcessing dataset {i+1}")
    
    # Get dataset name for reference
    name = adata.obs['orig.ident'].unique()[0] if 'orig.ident' in adata.obs else f"dataset_{i}"
    print(f"Dataset name: {name}")
    
    # Step 1: Run Leiden clustering if not already done
    if 'res_0.5' not in adata.obs:
        print("Running Leiden clustering...")
        sc.tl.leiden(adata, resolution=0.5, key_added="res_0.5", random_state=42)

    # Step 2: Find all '_pred' columns in .obs
    pred_columns = [col for col in adata.obs.columns if col.endswith('_pred')]
    print(f"Found {len(pred_columns)} '_pred' columns: {pred_columns}")

    # Step 3: Define UMAP key
    umap_key = 'X_umap'
    if umap_key not in adata.obsm:
        print(f"Warning: {umap_key} not found in obsm. Skipping this dataset.")
        continue

    # Step 4: Build umap_config dynamically — compare each pred column to Leiden clusters
    cluster_key = "res_0.5"
    umap_key = 'X_umap'  # Only one UMAP key
    
    # One UMAP key, multiple label/cluster pairs
    umap_config = {
        umap_key: [(col, cluster_key) for col in pred_columns]
    }

    print(f"Running metrics for UMAP '{umap_key}' with config:")
    for k, v in umap_config.items():
        print(f"  {k} -> {v}")

    # Step 5: Run the metric calculation
    try:
        metrics_df = calculate_bio_conservation(
            adata,
            umap_config,
            batch_key="orig.ident",
            compute_isolated_labels=False,
            subsample=None
        )

        # Add dataset name to results
        metrics_df['Dataset'] = name

        # Append to results list
        all_metrics_dfs.append(metrics_df)

    except Exception as e:
        print(f"Error processing dataset {name}: {e}")
        traceback.print_exc()

# Step 6: Combine all results into one DataFrame
if all_metrics_dfs:
    final_metrics_df = pd.concat(all_metrics_dfs, axis=0, ignore_index=True)
    print("\n✅ Final Metrics DataFrame:")
    print(final_metrics_df.head())
else:
    print("No metrics were calculated.")


Processing dataset 1
Dataset name: Weatherbee
Found 10 '_pred' columns: ['scpoli_lineage_pred', 'scpoli_reanno_pred', 'querymap_lineage_pred', 'querymap_reanno_pred', 'scgpt_lineage_pred', 'scgpt_reanno_pred', 'scArches_reanno_pred', 'scArches_lineage_pred', 'garfield_reanno_pred', 'garfield_lineage_pred']
Running metrics for UMAP 'X_umap' with config:
  X_umap -> [('scpoli_lineage_pred', 'res_0.5'), ('scpoli_reanno_pred', 'res_0.5'), ('querymap_lineage_pred', 'res_0.5'), ('querymap_reanno_pred', 'res_0.5'), ('scgpt_lineage_pred', 'res_0.5'), ('scgpt_reanno_pred', 'res_0.5'), ('scArches_reanno_pred', 'res_0.5'), ('scArches_lineage_pred', 'res_0.5'), ('garfield_reanno_pred', 'res_0.5'), ('garfield_lineage_pred', 'res_0.5')]
Processing X_umap with lineage: scpoli_lineage_pred, clusters: res_0.5
  Working with 5039 cells
  Silhouette score: 0.6313561052083969
  NMI score: 0.630951365964821
  ARI score: 0.3183806384241711
  Completed in 0.63 seconds
Processing X_umap with lineage: scpoli_

In [19]:
# Save results
final_metrics_df.to_csv('./dataset_scib_benchmarking.csv', index=True)

In [20]:
import scanpy as sc
import os

# Ensure the output directory exists
output_dir = "./umap_plots"
os.makedirs(output_dir, exist_ok=True)

# Loop through each AnnData in the list
for i, adata in enumerate(adata_list):
    print(f"\nPlotting UMAPs for dataset {i+1}")
    
    # Get dataset name
    name = adata.obs['orig.ident'].unique()[0] if 'orig.ident' in adata.obs else f"dataset_{i}"
    print(f"Dataset name: {name}")
    
    # Step 1: Find all '_pred' columns in .obs
    pred_columns = [col for col in adata.obs.columns if col.endswith('_pred')]
    print(f"Found {len(pred_columns)} '_pred' columns: {pred_columns}")
    
    # Step 2: Add res_0.5 to the list of columns to plot
    keys_to_plot = ['res_0.5'] + pred_columns
    
    # Step 3: Generate UMAP plots
    sc.settings.figdir = os.path.join(output_dir, name)
    os.makedirs(sc.settings.figdir, exist_ok=True)

    print(f"Saving UMAPs to: {sc.settings.figdir}")
    
    sc.pl.umap(
        adata,
        color=keys_to_plot,
        ncols=4,  # Adjust number of columns per figure
        show=False,
        save=f"_{name}.pdf"
    )

    print(f"Saved UMAPs for {name}")


Plotting UMAPs for dataset 1
Dataset name: Weatherbee
Found 10 '_pred' columns: ['scpoli_lineage_pred', 'scpoli_reanno_pred', 'querymap_lineage_pred', 'querymap_reanno_pred', 'scgpt_lineage_pred', 'scgpt_reanno_pred', 'scArches_reanno_pred', 'scArches_lineage_pred', 'garfield_reanno_pred', 'garfield_lineage_pred']
Saving UMAPs to: umap_plots/Weatherbee


/home/liuxiaodongLab/fanxueying/miniconda3/envs/benchmarking/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/liuxiaodongLab/fanxueying/miniconda3/envs/benchmarking/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/liuxiaodongLab/fanxueying/miniconda3/envs/benchmarking/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/liuxiaodongLab/fanxueying/miniconda3/envs/benchmarking/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/liuxiaodongLab/fanxueying/minicond

/home/liuxiaodongLab/fanxueying/miniconda3/envs/benchmarking/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/liuxiaodongLab/fanxueying/miniconda3/envs/benchmarking/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAPs for Weatherbee

Plotting UMAPs for dataset 2
Dataset name: Liu
Found 10 '_pred' columns: ['scpoli_lineage_pred', 'scpoli_reanno_pred', 'querymap_lineage_pred', 'querymap_reanno_pred', 'scgpt_lineage_pred', 'scgpt_reanno_pred', 'scArches_reanno_pred', 'scArches_lineage_pred', 'garfield_reanno_pred', 'garfield_lineage_pred']
Saving UMAPs to: umap_plots/Liu


/home/liuxiaodongLab/fanxueying/miniconda3/envs/benchmarking/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/liuxiaodongLab/fanxueying/miniconda3/envs/benchmarking/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/liuxiaodongLab/fanxueying/miniconda3/envs/benchmarking/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/liuxiaodongLab/fanxueying/miniconda3/envs/benchmarking/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/liuxiaodongLab/fanxueying/minicond

...

/home/liuxiaodongLab/fanxueying/miniconda3/envs/benchmarking/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/liuxiaodongLab/fanxueying/miniconda3/envs/benchmarking/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/liuxiaodongLab/fanxueying/miniconda3/envs/benchmarking/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAPs for Pedroza

Plotting UMAPs for dataset 9
Dataset name: zheng_2022
Found 10 '_pred' columns: ['scpoli_lineage_pred', 'scpoli_reanno_pred', 'querymap_lineage_pred', 'querymap_reanno_pred', 'scgpt_lineage_pred', 'scgpt_reanno_pred', 'scArches_reanno_pred', 'scArches_lineage_pred', 'garfield_reanno_pred', 'garfield_lineage_pred']
Saving UMAPs to: umap_plots/zheng_2022


/home/liuxiaodongLab/fanxueying/miniconda3/envs/benchmarking/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/liuxiaodongLab/fanxueying/miniconda3/envs/benchmarking/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/liuxiaodongLab/fanxueying/miniconda3/envs/benchmarking/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/liuxiaodongLab/fanxueying/miniconda3/envs/benchmarking/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/liuxiaodongLab/fanxueying/minicond

/home/liuxiaodongLab/fanxueying/miniconda3/envs/benchmarking/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/liuxiaodongLab/fanxueying/miniconda3/envs/benchmarking/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAPs for zheng_2022
